In [ ]:
!pip install gradio -q

import pandas as pd
import numpy as np
import re
import gradio as gr

In [ ]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])

print(df.head())
print(df.info())

Saving service_recommendation_dataset_100000.csv to service_recommendation_dataset_100000.csv
  Service_ID  Service_Category            Service_Name  \
0  SRV000001  Appliance Repair  ApplianceCare Services   
1  SRV000002         Carpenter     TimberWorks Experts   
2  SRV000003      Pest Control            PestOut Care   
3  SRV000004           Painter            PaintPro Co.   
4  SRV000005  Appliance Repair             FixMate Co.   

             Service_Type                             Problem_Description  \
0  Water Purifier Service         Water purifier filter needs replacement   
1            Door Fitting                Wooden door not closing properly   
2         Termite Control              Termites found in wooden furniture   
3           Waterproofing  Ceiling has damp patches needing waterproofing   
4        Chimney Cleaning              Chimney filter clogged with grease   

    Location  Distance_KM  Rating  Price           Availability  \
0   Palakkad        39.37  

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Numeric columns
numeric_columns = [
    "Distance_KM",
    "Rating",
    "Price",
    "Experience_Years"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Categorical columns
categorical_columns = [
    "Service_Category",
    "Service_Name",
    "Service_Type",
    "Problem_Description",
    "Location",
    "Availability"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Service_ID             0
Service_Category       0
Service_Name           0
Service_Type           0
Problem_Description    0
Location               0
Distance_KM            0
Rating                 0
Price                  0
Availability           0
Experience_Years       0
dtype: int64


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
service_keywords = {

    "Automobile": [
        "car", "bike", "vehicle", "scooter",
        "motorcycle", "engine", "tyre", "tire",
        "battery", "brake", "oil", "mechanic",
        "car repair", "bike repair"
    ],

    "Plumber": [
        "pipe", "water", "tap", "leak",
        "leakage", "bathroom", "sink",
        "plumbing", "plumber"
    ],

    "Electrician": [
        "electric", "electricity", "wire",
        "wiring", "switch", "fan", "light",
        "socket", "power", "current",
        "electrician"
    ],

    "AC Technician": [
        "ac", "air conditioner", "cooling",
        "air conditioning", "ac repair"
    ],

    "Appliance Repair": [
        "washing machine", "fridge",
        "refrigerator", "microwave",
        "television", "tv", "appliance"
    ],

    "Computer Repair": [
        "computer", "laptop", "keyboard",
        "mouse", "printer", "software",
        "pc", "computer repair"
    ]
}

In [ ]:
def detect_service(query):

    query = clean_text(query)

    scores = {}

    for category, keywords in service_keywords.items():

        score = 0

        for keyword in keywords:

            keyword = clean_text(keyword)

            if keyword in query:
                score += 1

        scores[category] = score

    best_category = max(scores, key=scores.get)

    if scores[best_category] == 0:
        return None

    return best_category

In [ ]:
locations = df["Location"].astype(str).unique().tolist()

def detect_location(query):

    query = clean_text(query)

    for location in locations:

        location_clean = clean_text(location)

        if location_clean in query:
            return location

    return None

In [ ]:
def recommend_services(category, location=None):

    result = df[
        df["Service_Category"].astype(str).str.lower()
        == category.lower()
    ].copy()

    # Filter location if detected
    if location is not None:

        location_result = result[
            result["Location"].astype(str).str.lower()
            == location.lower()
        ]

        # Use location results if available
        if len(location_result) > 0:
            result = location_result

    if len(result) == 0:
        return pd.DataFrame()

    # Normalized scores

    max_distance = result["Distance_KM"].max()

    if max_distance == 0:
        max_distance = 1

    max_price = result["Price"].max()

    if max_price == 0:
        max_price = 1

    max_experience = result["Experience_Years"].max()

    if max_experience == 0:
        max_experience = 1

    # Recommendation score

    rating_score = result["Rating"] / 5

    distance_score = (
        1 - result["Distance_KM"] / max_distance
    )

    price_score = (
        1 - result["Price"] / max_price
    )

    experience_score = (
        result["Experience_Years"] / max_experience
    )

    availability_score = result["Availability"].apply(
        lambda x: 1 if str(x).lower() == "available" else 0.5
    )

    result["Recommendation_Score"] = (

        rating_score * 0.35 +

        distance_score * 0.20 +

        price_score * 0.15 +

        experience_score * 0.20 +

        availability_score * 0.10
    )

    result = result.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result.head(5)

In [ ]:
def solve_problem(query):

    if query is None or query.strip() == "":
        return "Please describe your problem.", pd.DataFrame()

    # Detect service
    category = detect_service(query)

    if category is None:
        return (
            "❌ I could not identify the required service.",
            pd.DataFrame()
        )

    # Detect location
    location = detect_location(query)

    # Get recommendations
    result = recommend_services(
        category,
        location
    )

    if len(result) == 0:
        return (
            f"Service detected: {category}\n"
            "No matching service providers found.",
            pd.DataFrame()
        )

    message = f"🔍 Service Detected: {category}"

    if location:
        message += f"\n📍 Location Detected: {location}"

    return message, result[
        [
            "Service_Name",
            "Service_Type",
            "Service_Category",
            "Location",
            "Distance_KM",
            "Rating",
            "Price",
            "Availability",
            "Experience_Years",
            "Recommendation_Score"
        ]
    ]

In [ ]:
custom_css = """
body {
    background: linear-gradient(135deg, #dbeafe, #f0fdf4);
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto;
}

h1 {
    text-align: center;
    color: #1e3a8a;
}

textarea {
    border-radius: 15px !important;
}

button {
    border-radius: 12px !important;
    font-size: 16px !important;
}
"""

with gr.Blocks(
    css=custom_css,
    title="AI Multi-Service Finder"
) as app:

    gr.Markdown(
        """
        # 🤖 AI Multi-Service Finder

        ### Describe your problem and let AI find the right service.
        """
    )

    query = gr.Textbox(
        label="Describe Your Problem",
        placeholder=
        "Example: My bike is not starting and I need a mechanic near Perinthalmanna.",
        lines=4
    )

    search_button = gr.Button(
        "🔍 Find Best Service",
        variant="primary"
    )

    status = gr.Textbox(
        label="AI Analysis"
    )

    results = gr.Dataframe(
        label="Recommended Service Providers",
        interactive=False
    )

    search_button.click(
        fn=solve_problem,
        inputs=query,
        outputs=[status, results]
    )

    gr.Markdown(
        """
        ### 💡 Example queries

        - My bike is not starting near Perinthalmanna.
        - There is a water pipe leakage in my house.
        - My AC is not cooling properly.
        - I have an electrical wiring problem.
        - My washing machine is not working.
        - My laptop is not turning on.
        """
    )

app.launch()

/tmp/ipykernel_1084/2365879641.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7e66d4b7944f9904f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
